# Conteo de registros H y M

Sistemas de Big Data 5074 · UT1.2 · Diapositivas 1–11.

Versión para completar. Puedes trabajar en este cuaderno en lugar de `alumno.py`. Sustituye los `NotImplementedError` por tu código. Las celdas con tareas pendientes se detendrán hasta que las completes.

Solo biblioteca estándar de Python 3.10 o posterior. Los datos están incluidos. No requiere instalaciones ni archivos externos. Puedes subir este `.ipynb` a Colab. Los ejemplos MapReduce se simulan localmente y en serie, sin ejecutar Hadoop.


# Conteo de registros por categoría H/M

El objetivo consiste en contar cuántos registros contienen la etiqueta H y cuántos la etiqueta M. La categoría se lee del último campo. El nombre no determina esa etiqueta.

Trabaja en `alumno.py`. Necesitas Python 3.10 o posterior y su biblioteca estándar. Los imports y los datos ya están preparados. Sustituye cada `NotImplementedError` por tu implementación. Ver ese error antes de completar una función es normal en esta plantilla.

## Datos y alcance

Hay tres ficheros con tres registros cada uno. Los tres registros de `1800.txt` reproducen el ejemplo visible de la diapositiva 3. Los otros seis registros son ficticios, creados para esta práctica. El resultado de los nueve registros corresponde a esta práctica, no a un conjunto completo mostrado en la presentación.

`FICHEROS` contiene los datos dentro del propio programa. Los archivos de `datos/` son su copia en UTF-8 con saltos de línea LF. No necesitas leerlos del disco para ejecutar el programa.

Cada fichero representa un split lógico didáctico. El recorrido se ejecutará de forma local y serial. H/M son las etiquetas de este conjunto didáctico. Contamos apariciones de registros, sin deduplicar personas entre ficheros.

## Trabajo paso a paso

1. Lee los nueve registros y calcula a mano el resultado esperado. Anota qué campo debe ser la clave intermedia para poder sumar por categoría.
2. Completa `map_registro`. Separa los campos y comprueba que hay un nombre no vacío, una sola coma y una categoría válida. Emite un par por registro. Pistas: `rpartition` y `strip`.
3. Completa `shuffle`. Recorre los pares emitidos y construye una lista de valores por clave. Conserva todos los valores repetidos. Pista: `defaultdict`.
4. Completa `reduce_categoria`. Escribe un bucle que acumule los valores de un grupo y devuelve el par de salida. Después contrasta tu acumulación con `sum`.
5. Completa `ejecutar_mapreduce`. Escribe los bucles que recorren los ficheros y sus líneas, recoge las emisiones, agrupa y reduce. Si un registro falla, informa del fichero y la línea. Pistas: `splitlines` y `enumerate`.
6. Completa `conteo_secuencial` con un recorrido directo que actualice los contadores sin construir emisiones ni grupos.
7. Completa `mostrar_demostracion`: muestra las nueve emisiones Map, los grupos y los dos resultados Reduce. Compara los métodos y comprueba que no se pierde ningún registro.

## Evidencias que debes obtener

- Nueve emisiones Map, una por registro.
- Dos grupos, uno por categoría, con todos sus valores.
- Un diccionario final con el conteo que hayas calculado para cada categoría.
- El mismo resultado mediante el conteo secuencial.
- Igualdad entre registros, emisiones Map y valores agrupados. Comprueba que `H + M = 9` usando los conteos que hayas obtenido.

Un registro vacío, uno sin coma, uno sin nombre o uno con categoría distinta de H/M debe provocar un error claro. No lo ignores silenciosamente. Los saltos LF que cierran los registros no forman un registro adicional.

## Relación con la presentación

Las diapositivas 7–9 usan el nombre como clave intermedia. La diapositiva 10 compara después la clave con H/M. Explica por qué esas dos decisiones no encajan. Para este objetivo la clave intermedia es la categoría, que es precisamente aquello por lo que se quiere contar.

## Qué representa la simulación

Las funciones muestran la transformación Map, la agrupación por clave y la reducción. El programa no ejecuta Hadoop ni reproduce HDFS, la transferencia por red, collect, spill, merge, planificación o tolerancia a fallos. No se deben extraer conclusiones de rendimiento distribuido a partir de esta ejecución local.


## Código 1 Presentación del bloque


In [32]:
"""Plantilla intermedia: conteo de registros por categoría H/M.

Python 3.10 o posterior. Solo biblioteca estándar.
Completa las funciones sustituyendo cada NotImplementedError.
Los tres primeros registros son los del ejemplo de la diapositiva 3.
Los otros seis son ficticios y se han añadido para esta práctica.
Cada fichero representa un split lógico didáctico. La ejecución es serial
y local. Esta plantilla no ejecuta Hadoop.
"""


'Plantilla intermedia: conteo de registros por categoría H/M.\n\nPython 3.10 o posterior. Solo biblioteca estándar.\nCompleta las funciones sustituyendo cada NotImplementedError.\nLos tres primeros registros son los del ejemplo de la diapositiva 3.\nLos otros seis son ficticios y se han añadido para esta práctica.\nCada fichero representa un split lógico didáctico. La ejecución es serial\ny local. Esta plantilla no ejecuta Hadoop.\n'

## Código 2 Imports y datos preparados


In [33]:
from collections import defaultdict


FICHEROS: dict[str, str] = {
    "1800.txt": (
        "Manuel García García, H\n"
        "Juan Belmonte Pérez, H\n"
        "María Selas Rodríguez, M\n"
    ),
    "1801.txt": (
        "Lucía Martín Pérez, M\n"
        "Diego Ramos Díaz, H\n"
        "Elena Suárez León, M\n"
    ),
    "1802.txt": (
        "Pablo Torres Gil, H\n"
        "Nora Medina Ruiz, M\n"
        "Iván Cabrera Soto, H\n"
    ),
}

CATEGORIAS = ("H", "M")


## Código 3 Map: parseo, validación y emisión


In [34]:
def map_registro(linea: str) -> tuple[str, int]:
    """Recibe una línea sin terminador y emite (categoría, 1).

    TODO: separa nombre y categoría. Rechaza nombres vacíos, registros
    sin una única coma, saltos de línea internos y categorías inválidas.
    Lee H/M del campo final. No deduzcas la categoría a partir del nombre.
    Pista: investiga rpartition y strip.
    """
    if "\n" in linea or "\r" in linea:
        raise ValueError("Salto de línea interno")
    if linea.count(",") != 1:
        raise ValueError("El registro debe tener una única coma")
    nombre, _, categoria = linea.rpartition(",")
    nombre = nombre.strip()
    categoria = categoria.strip()
    if not nombre:
        raise ValueError("Nombre vacío")
    if categoria not in CATEGORIAS:
        raise ValueError(f"Categoría inválida: {categoria!r}")
    return (categoria, 1)

map_registro("Manuel García García, H") 

('H', 1)

## Código 4 Shuffle: agrupación sin perder repeticiones


In [35]:
def shuffle(pares: list[tuple[str, int]]) -> dict[str, list[int]]:
    """Recibe emisiones Map y devuelve una lista de valores por clave.

    TODO: escribe el recorrido de pares y la construcción de los grupos.
    Cada 1 debe conservarse, aunque otros valores sean iguales.
    Pista: defaultdict puede facilitar la creación de listas vacías.
    """
    grupos = defaultdict(list)
    for clave, valor in pares:
        grupos[clave].append(valor)
    return dict(grupos)


## Código 5 Reduce: cálculo por categoría


In [36]:
def reduce_categoria(clave: str, valores: list[int]) -> tuple[str, int]:
    """Devuelve (categoría, total) para un único grupo.

    TODO: escribe la acumulación de los valores y devuelve el par de salida.
    Pista: resuélvelo primero con un bucle. Después puedes compararlo con sum.
    """
    total = 0
    for v in valores:
        total += v
    assert total == sum(valores)
    return (clave, total)


## Código 6 Coordinación de las fases


In [37]:
def ejecutar_mapreduce(ficheros: dict[str, str]) -> dict[str, int]:
    """Devuelve el conteo de las categorías presentes en los registros.

    TODO: recorre los ficheros y sus líneas, llama a Map y recoge sus pares.
    Después agrupa y aplica Reduce a cada clave. Construye el resultado.
    Los errores deben indicar fichero y número de línea, sin ignorarlos.
    Pista: splitlines, enumerate y los contratos de las funciones anteriores.
    """
    pares = []
    for nombre, contenido in ficheros.items():
        for n, linea in enumerate(contenido.splitlines(), start=1):
            try:
                pares.append(map_registro(linea))
            except ValueError as e:
                raise ValueError(f"{nombre}, línea {n}: {e}") from e

    grupos = shuffle(pares)
    resultado = {}
    for clave, valores in grupos.items():
        cat, total = reduce_categoria(clave, valores)
        resultado[cat] = total
    return resultado


## Código 7 Contraste secuencial


In [38]:
def conteo_secuencial(ficheros: dict[str, str]) -> dict[str, int]:
    """Cuenta directamente los registros H/M sin crear listas de emisiones.

    TODO: recorre y valida los datos. Actualiza un contador por categoría.
    Este recorrido permitirá contrastar tu resultado MapReduce.
    """
    contadores = {c: 0 for c in CATEGORIAS}
    for nombre, contenido in ficheros.items():
        for n, linea in enumerate(contenido.splitlines(), start=1):
            try:
                cat, _ = map_registro(linea)
            except ValueError as e:
                raise ValueError(f"{nombre}, línea {n}: {e}") from e
            contadores[cat] += 1
    return {k: v for k, v in contadores.items() if v}


## Código 8 Mostrar las fases y comprobar el resultado


In [39]:
def mostrar_demostracion() -> None:
    """Construye una traza de los datos preparados.

    TODO: muestra una emisión por registro, los grupos y cada salida Reduce.
    Ejecuta ambos métodos y comprueba su igualdad. Comprueba también que el
    número de registros, emisiones y valores agrupados coincide con H + M.
    Calcula los conteos de H y M. Su suma debe ser igual a los 9 registros.
    """
    total_registros = sum(len(c.splitlines()) for c in FICHEROS.values())

    pares = []
    for nombre, contenido in FICHEROS.items():
        for n, linea in enumerate(contenido.splitlines(), start=1):
            try:
                pares.append(map_registro(linea))
            except ValueError as e:
                raise ValueError(f"{nombre}, línea {n}: {e}") from e

    print("== Map ==")
    for p in pares:
        print(p)

    grupos = shuffle(pares)
    print("\nShuffle")
    for k, v in grupos.items():
        print(k, v)

    print("\nReduce")
    salidas = [reduce_categoria(k, v) for k, v in grupos.items()]
    for s in salidas:
        print(s)

    mr = ejecutar_mapreduce(FICHEROS)
    sec = conteo_secuencial(FICHEROS)
    print("\nMapReduce:", mr)
    print("Secuencial:", sec)

    valores_agrupados = sum(len(v) for v in grupos.values())
    assert mr == sec, "Los métodos no coinciden"
    assert total_registros == len(pares) == valores_agrupados == 9
    assert mr["H"] + mr["M"] == total_registros
    print(f"\nOK: H={mr['H']} + M={mr['M']} = {total_registros} registros")


## Código 9 Ejecutar tu demostración


In [40]:
if __name__ == "__main__":
    mostrar_demostracion()


== Map ==
('H', 1)
('H', 1)
('M', 1)
('M', 1)
('H', 1)
('M', 1)
('H', 1)
('M', 1)
('H', 1)

Shuffle
H [1, 1, 1, 1, 1]
M [1, 1, 1, 1]

Reduce
('H', 5)
('M', 4)

MapReduce: {'H': 5, 'M': 4}
Secuencial: {'H': 5, 'M': 4}

OK: H=5 + M=4 = 9 registros


## Explicación de tus resultados

El resultado es H = 5 y M = 4, con un total de 9 registros. Hay 9 emisiones Map, 2 grupos (H con 5 valores y M con 4) y el conteo secuencial da lo mismo. Así se comprueba que no se pierde ningún registro.
